# 📓 Day 3 Assignment: Library Management System Solution

Welcome to the reference solution notebook for the Day 3 Assignment. This notebook provides a modular, production-grade, object-oriented implementation of the **Coimbatore Municipal Library Catalog & Loan System** CLI, satisfying all core specifications and including **all bonus challenges** (CSV exporting and overdue date calculations).

## 1. Domain Custom Exceptions
We declare custom exceptions inheriting from a base `LibraryError` class to handle validation check violations in our business logic.

In [ ]:
class LibraryError(Exception):
    """Base exception for all library system anomalies."""
    pass

class BookNotFoundError(LibraryError):
    """Raised when searching for an ISBN not cataloged."""
    pass

class BookUnavailableError(LibraryError):
    """Raised when borrowing a book with 0 available stock."""
    pass

class MemberNotFoundError(LibraryError):
    """Raised when referencing a Member ID not in registry."""
    pass

class MemberLimitExceededError(LibraryError):
    """Raised when a member attempts to borrow more than 3 books."""
    pass

## 2. Book & Member Model Classes
Here we declare the `Book` and `Member` classes. Note the use of private variables (`__borrowed_count`) and serialization mappings (`to_dict`).

In [ ]:
class Book:
    def __init__(self, isbn: str, title: str, author: str, quantity: int, borrowed_count: int = 0):
        self.isbn = isbn.strip()
        self.title = title.strip()
        self.author = author.strip()
        self.quantity = quantity
        # Encapsulation: Private tracker of active loans
        self.__borrowed_count = borrowed_count

    def get_borrowed_count(self) -> int:
        return self.__borrowed_count

    def get_available_copies(self) -> int:
        return self.quantity - self.__borrowed_count

    def borrow_copy(self):
        if self.get_available_copies() <= 0:
            raise BookUnavailableError(f"No copies of '{self.title}' are available for checkout.")
        self.__borrowed_count += 1

    def return_copy(self):
        if self.__borrowed_count <= 0:
            raise ValueError(f"System error: Borrowed count for '{self.title}' is already zero.")
        self.__borrowed_count -= 1

    def to_dict(self) -> dict:
        return {
            "isbn": self.isbn,
            "title": self.title,
            "author": self.author,
            "quantity": self.quantity,
            "borrowed_count": self.__borrowed_count
        }

    def __str__(self) -> str:
        return f"'{self.title}' by {self.author} (ISBN: {self.isbn}) | Available: {self.get_available_copies()}/{self.quantity}"


class Member:
    def __init__(self, member_id: str, name: str, borrowed_books: list = None):
        self.member_id = member_id.strip()
        self.name = name.strip()
        # List of dicts: [{'isbn': '...', 'checkout_date': 'YYYY-MM-DD'}]
        self.borrowed_books = borrowed_books if borrowed_books is not None else []

    def borrow_book(self, isbn: str, checkout_date: str):
        if len(self.borrowed_books) >= 3:
            raise MemberLimitExceededError(f"Member {self.name} reached checkout limit of 3 books.")
        self.borrowed_books.append({
            "isbn": isbn,
            "checkout_date": checkout_date
        })

    def return_book(self, isbn: str) -> str:
        """Removes book from member profile list and returns its checkout_date string."""
        for idx, record in enumerate(self.borrowed_books):
            if record["isbn"] == isbn:
                checkout_date = record["checkout_date"]
                self.borrowed_books.pop(idx)
                return checkout_date
        raise ValueError(f"Member {self.name} did not check out ISBN: {isbn}.")

    def to_dict(self) -> dict:
        return {
            "member_id": self.member_id,
            "name": self.name,
            "borrowed_books": self.borrowed_books
        }

    def __str__(self) -> str:
        book_count = len(self.borrowed_books)
        return f"Member: {self.name} (ID: {self.member_id}) | Active Loans: {book_count}/3"

## 3. Library Management Engine Class
The core orchestrator handling JSON file persistent states, transactions, and CSV exports.

In [ ]:
import json
import csv
import os
from datetime import datetime

class Library:
    def __init__(self, db_file: str = "library_db.json"):
        self.db_file = db_file
        self.books = {}    # isbn -> Book object
        self.members = {}  # member_id -> Member object
        self.load_data()

    def load_data(self):
        if not os.path.exists(self.db_file):
            return
        try:
            with open(self.db_file, "r") as f:
                raw_data = json.load(f)
                
                # Reconstruct Book objects
                for isbn, item in raw_data.get("books", {}).items():
                    self.books[isbn] = Book(
                        isbn=item["isbn"],
                        title=item["title"],
                        author=item["author"],
                        quantity=item["quantity"],
                        borrowed_count=item.get("borrowed_count", 0)
                    )
                
                # Reconstruct Member objects
                for m_id, item in raw_data.get("members", {}).items():
                    self.members[m_id] = Member(
                        member_id=item["member_id"],
                        name=item["name"],
                        borrowed_books=item.get("borrowed_books", [])
                    )
        except (json.JSONDecodeError, IOError) as e:
            print(f"⚠️ Error loading database file: {e}. Starting with a blank memory state.")

    def save_data(self):
        try:
            db_snapshot = {
                "books": {isbn: b.to_dict() for isbn, b in self.books.items()},
                "members": {m_id: m.to_dict() for m_id, m in self.members.items()}
            }
            with open(self.db_file, "w") as f:
                json.dump(db_snapshot, f, indent=4)
        except IOError as e:
            print(f"❌ Critical: Failed to save changes to disk: {e}")

    def add_book(self, isbn: str, title: str, author: str, quantity: int):
        isbn = isbn.strip()
        if isbn in self.books:
            # Update quantity
            self.books[isbn].quantity += quantity
            print(f"Updated stock count for existing book: {self.books[isbn]}")
        else:
            self.books[isbn] = Book(isbn, title, author, quantity)
            print(f"✅ Successfully registered new book: '{title}'")
        self.save_data()

    def register_member(self, member_id: str, name: str):
        member_id = member_id.strip()
        if member_id in self.members:
            print(f"⚠️ Registration Warning: Member ID '{member_id}' already registered.")
            return
        self.members[member_id] = Member(member_id, name)
        print(f"✅ Successfully registered member: {name}")
        self.save_data()

    def search_books(self, query: str) -> list:
        query_lower = query.lower().strip()
        matches = []
        for book in self.books.values():
            if query_lower in book.title.lower() or query_lower in book.author.lower():
                matches.append(book)
        return matches

    def checkout_book(self, member_id: str, isbn: str, date_str: str = None):
        member_id = member_id.strip()
        isbn = isbn.strip()
        
        if date_str is None:
            date_str = datetime.now().strftime("%Y-%m-%d")
            
        # Validations
        if member_id not in self.members:
            raise MemberNotFoundError(f"ID '{member_id}' not found in member database.")
        if isbn not in self.books:
            raise BookNotFoundError(f"ISBN '{isbn}' not registered in book database.")
            
        member = self.members[member_id]
        book = self.books[isbn]
        
        # Check if member already has this specific book checked out
        for item in member.borrowed_books:
            if item["isbn"] == isbn:
                raise LibraryError(f"Member '{member.name}' has already checked out a copy of '{book.title}'.")
                
        # Process checkout
        book.borrow_copy() # Might raise BookUnavailableError
        member.borrow_book(isbn, date_str) # Might raise MemberLimitExceededError
        
        self.save_data()
        print(f"🎉 Successful checkout! '{book.title}' issued to {member.name}.")

    def process_return(self, member_id: str, isbn: str, return_date_str: str = None) -> float:
        """Processes book returns and returns calculated overdue fines."""
        member_id = member_id.strip()
        isbn = isbn.strip()
        
        if return_date_str is None:
            return_date_str = datetime.now().strftime("%Y-%m-%d")
            
        if member_id not in self.members:
            raise MemberNotFoundError(f"ID '{member_id}' not found in member database.")
        if isbn not in self.books:
            raise BookNotFoundError(f"ISBN '{isbn}' not registered in book database.")
            
        member = self.members[member_id]
        book = self.books[isbn]
        
        # Process return and get borrow date
        try:
            checkout_date_str = member.return_book(isbn)
        except ValueError:
            raise LibraryError(f"Member '{member.name}' does not have active checkout for ISBN '{isbn}'.")
            
        book.return_copy()
        self.save_data()
        
        # Fine Calculation Logic (Bonus Challenge)
        fine = 0.0
        try:
            d1 = datetime.strptime(checkout_date_str, "%Y-%m-%d")
            d2 = datetime.strptime(return_date_str, "%Y-%m-%d")
            borrow_duration_days = (d2 - d1).days
            if borrow_duration_days > 14:
                overdue_days = borrow_duration_days - 14
                fine = float(overdue_days * 5)
                print(f"⚠️ Return Overdue by {overdue_days} days! Fine incurred: ₹{fine:.2f}")
            else:
                print(f"✅ Returned in {borrow_duration_days} days. No fine.")
        except ValueError:
            # Fallback if dates were input in incorrect formats
            print("⚠️ Could not parse dates for fine calculation. Fine set to zero.")
            
        print(f"💸 Book '{book.title}' returned successfully by {member.name}.")
        return fine

    def export_loans_csv(self, filepath: str = "active_loans.csv"):
        try:
            loans_found = False
            with open(filepath, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["Member ID", "Member Name", "Book Title", "ISBN", "Checkout Date"])
                
                for member in self.members.values():
                    for loan in member.borrowed_books:
                        book_title = self.books.get(loan["isbn"], Book("0000", "Unknown", "Unknown", 0)).title
                        writer.writerow([
                            member.member_id,
                            member.name,
                            book_title,
                            loan["isbn"],
                            loan["checkout_date"]
                        ])
                        loans_found = True
            if loans_found:
                print(f"📊 Active loan records exported successfully to: {filepath}")
            else:
                print("ℹ️ No active loans currently exist to export.")
        except IOError as e:
            print(f"❌ Failed to export CSV: {e}")

## 4. Interactive CLI Wrapper
The console user interface helper wrapping the input routines inside safety try-except catch loops.

In [ ]:
def get_safe_int_input(prompt: str) -> int:
    while True:
        try:
            val = int(input(prompt))
            if val < 0:
                print("❌ Error: Please enter a positive number.")
                continue
            return val
        except ValueError:
            print("❌ Error: Invalid input formatting! Please enter whole integers only.")

def run_library_cli():
    lib = Library("library_db.json")
    
    while True:
        print("\n======================================")
        print("     COIMBATORE PUBLIC LIBRARY")
        print("======================================")
        print("1. Book Catalog Operations")
        print("2. Member Registry Operations")
        print("3. Issue & Return Transactions")
        print("4. Export Active Loans Report (CSV)")
        print("5. Save & Exit")
        print("======================================")
        choice = input("Enter choice (1-5): ").strip()
        
        if choice == "1":
            while True:
                print("\n--- BOOK CATALOG OPERATIONS ---")
                print("1. Add Book")
                print("2. Search Book")
                print("3. List All Books")
                print("4. Back to Main Menu")
                sub_choice = input("Select option (1-4): ").strip()
                
                if sub_choice == "1":
                    isbn = input("Enter ISBN: ").strip()
                    title = input("Enter Title: ").strip()
                    author = input("Enter Author: ").strip()
                    qty = get_safe_int_input("Enter Quantity: ")
                    if isbn and title and author:
                        lib.add_book(isbn, title, author, qty)
                    else:
                        print("❌ Add Failed: Title, Author, and ISBN cannot be blank.")
                
                elif sub_choice == "2":
                    query = input("Search keyword (Title/Author): ").strip()
                    if query:
                        results = lib.search_books(query)
                        print(f"\nMatches found ({len(results)}):")
                        for book in results:
                            print(f"- {book}")
                    else:
                        print("❌ Search query cannot be blank.")
                
                elif sub_choice == "3":
                    print("\n--- REGISTERED BOOKS IN CATALOG ---")
                    if not lib.books:
                        print("Catalog is currently empty.")
                    for book in lib.books.values():
                        print(f"- {book}")
                
                elif sub_choice == "4":
                    break
                    
        elif choice == "2":
            while True:
                print("\n--- MEMBER REGISTRY OPERATIONS ---")
                print("1. Register Member")
                print("2. List All Members")
                print("3. Back to Main Menu")
                sub_choice = input("Select option (1-3): ").strip()
                
                if sub_choice == "1":
                    m_id = input("Enter Member ID: ").strip()
                    name = input("Enter Full Name: ").strip()
                    if m_id and name:
                        lib.register_member(m_id, name)
                    else:
                        print("❌ Register Failed: ID and Name cannot be blank.")
                
                elif sub_choice == "2":
                    print("\n--- REGISTERED MEMBERS ---")
                    if not lib.members:
                        print("No registered members.")
                    for m in lib.members.values():
                        print(f"- {m}")
                
                elif sub_choice == "3":
                    break
                    
        elif choice == "3":
            while True:
                print("\n--- ISSUE & RETURN TRANSACTIONS ---")
                print("1. Checkout Book")
                print("2. Return Book")
                print("3. Back to Main Menu")
                sub_choice = input("Select option (1-3): ").strip()
                
                if sub_choice == "1":
                    m_id = input("Enter Member ID: ").strip()
                    isbn = input("Enter Book ISBN: ").strip()
                    date_in = input("Enter checkout date (YYYY-MM-DD or blank for today): ").strip()
                    date_arg = date_in if date_in else None
                    
                    try:
                        lib.checkout_book(m_id, isbn, date_arg)
                    except LibraryError as e:
                        print(f"❌ Checkout Denied: {e}")
                        
                elif sub_choice == "2":
                    m_id = input("Enter Member ID: ").strip()
                    isbn = input("Enter Book ISBN: ").strip()
                    date_in = input("Enter return date (YYYY-MM-DD or blank for today): ").strip()
                    date_arg = date_in if date_in else None
                    
                    try:
                        lib.process_return(m_id, isbn, date_arg)
                    except LibraryError as e:
                        print(f"❌ Return Denied: {e}")
                        
                elif sub_choice == "3":
                    break
                    
        elif choice == "4":
            lib.export_loans_csv("active_loans.csv")
            
        elif choice == "5":
            print("💾 Closing system libraries. Session saved successfully.")
            break
        else:
            print("⚠️ Invalid input. Select 1 to 5.")

## 5. Automated Verification Simulation
Let's execute a verification routine using mock assertions to verify that all requirements, including custom exceptions, inventory constraints, file save/load structures, and fine calculators work perfectly without starting the CLI.

In [ ]:
# Setup clean test database environment
test_db = "test_library.json"
if os.path.exists(test_db):
    os.remove(test_db)

try:
    test_lib = Library(test_db)
    
    print("--- 1. Testing Catalog Additions ---")
    test_lib.add_book("123-A", "Intro to Python", "Guido", quantity=2)
    test_lib.add_book("456-B", "Advanced Git", "Linus", quantity=1)
    test_lib.register_member("M001", "Amit")
    test_lib.register_member("M002", "Sneha")
    
    print("\n--- 2. Testing Checkout Boundaries ---")
    # Successful checkout
    test_lib.checkout_book("M001", "123-A", "2026-06-01")
    
    # Over limit checkout checks
    try:
        # Amit tries to checkout again. Should succeed because limit is 3
        test_lib.checkout_book("M001", "456-B", "2026-06-01")
    except LibraryError as e:
        print(f"Caught expected exception: {e}")
        
    print("\n--- 3. Testing Out of Stock Boundaries ---")
    # Try borrowing Git book when stock is already 0 (Linus's book quantity was 1, checked out by Amit)
    try:
        test_lib.checkout_book("M002", "456-B", "2026-06-01")
    except BookUnavailableError as e:
        print(f"✅ Successfully caught out-of-stock exception: {e}")

    print("\n--- 4. Testing Member Limits (Limit = 3) ---")
    # Add more books and make Amit borrow a 3rd and 4th book
    test_lib.add_book("789-C", "Book Three", "Author X", 2)
    test_lib.add_book("999-D", "Book Four", "Author Y", 2)
    
    test_lib.checkout_book("M001", "789-C", "2026-06-01") # 3rd book -> OK
    try:
        test_lib.checkout_book("M001", "999-D", "2026-06-01") # 4th book -> FAIL
    except MemberLimitExceededError as e:
        print(f"✅ Successfully caught limit exceeded exception: {e}")
        
    print("\n--- 5. Testing Overdue Return Fines (Borrow: Jun 1, Return: Jun 20 = 19 days. Overdue = 5 days -> Fine: ₹25) ---")
    fine = test_lib.process_return("M001", "123-A", "2026-06-20")
    assert fine == 25.0, f"Expected fine ₹25.0, got ₹{fine}"
    print(f"Fine assertion validation passed! Expected: ₹25.00, Calculated: ₹{fine:.2f}")
    
    print("\n--- 6. Testing Data Reloading Persistence ---")
    # Instantiating a new Library object targeting the same json file. It should restore state.
    restored_lib = Library(test_db)
    assert "123-A" in restored_lib.books, "Failed to restore book catalog."
    assert "M001" in restored_lib.members, "Failed to restore member registry."
    print(f"State reload check passed! Books loaded: {len(restored_lib.books)}, Members: {len(restored_lib.members)}")
    
    print("\n--- 7. Testing CSV Export ---")
    test_lib.export_loans_csv("test_active_loans.csv")
    assert os.path.exists("test_active_loans.csv"), "CSV file was not created."
    print("CSV export check passed!")
    
(

In [ ]:
# Cleanup test database files
for path in [test_db, "test_active_loans.csv"]:
    if os.path.exists(path):
        os.remove(path)
print("🧹 Test suite database environment cleaned up.")